### Pong – Lösung Schritt für Schritt

*33. Bundeswettbewerb Informatik (2014), 1. Runde, Aufgabe 5*

In dem alten Konsolenspiel Pong steuert jeder der beiden Spieler einen Schläger. Der Ball prallt sowohl von den Schlägern als auch von den oberen und unteren Spielfeldbegrenzungen ab. Verfehlt man den Ball mit dem eigenen Schläger, bekommt der Gegner einen Punkt.

Im BWINF-Turniersystem ist eine Pong-Variante implementiert: Das Spielfeld besteht aus horizontal 65 mal vertikal 60 quadratischen Zellen. Die Schläger befinden sich in der ganz linken bzw. ganz rechten Spalte und belegen je 6 zusammenhängende Zellen. Der Ball hat die Größe einer Zelle. Oben und unten wird das Spielfeld durch Banden begrenzt, von denen der Ball nach der Regel „Einfallswinkel = Ausfallswinkel“ abprallt.

Wie der Ball von einem Schläger abprallt, hängt davon ab, welches Drittel des Schlägers er trifft:

- Trifft der Ball das mittlere Drittel des Schlägers, prallt er nach der Regel „Einfallswinkel = Ausfallswinkel“ ab.
- Trifft der Ball das obere Drittel, prallt er in eine zufällige Richtung zwischen dem regulären Ausfallswinkel (wie oben) und 70° ab.
- Trifft der Ball das untere Drittel, prallt er in eine zufällige Richtung zwischen −70° und dem regulären Ausfallswinkel ab.

Man kann dem Gegner eine Überraschung bescheren, indem man sich im letzten Moment entscheidet, den Ball nicht mit der Mitte, sondern mit einem äußeren Drittel anzunehmen.

#### Aufgabe 5

Programmiere einen Spieler für Pong, der den linken Schläger steuert.

Das Spiel wird in Schritten gespielt. In jedem Schritt darfst du den Schläger stehen lassen oder um eine Zelle nach oben oder unten bewegen. Dabei sieht dein Programm das komplette Spielfeld, also die Positionen der beiden Schläger sowie die Zelle, auf der der Ball dargestellt wird. Beachte, dass der Ball sich kontinuierlich bewegt und daher die sichtbare Position seine tatsächliche Position nur annähert. Zu Beginn bewegt sich der Ball mit einer Geschwindigkeit von einer Zellenseitenlänge pro Schritt. Die Geschwindigkeit wird im Laufe des Spiels immer weiter gesteigert.

Beim Einwurf stehen beide Schläger in der Mitte, und der Ball startet vom Schläger eines Spielers in einem zufälligen Winkel. Sobald ein Punkt erzielt wurde, wird der Ball neu eingeworfen. Der Einwurf passiert abwechselnd von links und rechts.

Ein Spiel besteht aus zwei Halbzeiten. Jeder Spieler bekommt in einer Halbzeit den ersten Einwurf. Eine Halbzeit besteht aus einer festgelegten Zahl von Schritten. Der Spieler mit den meisten Punkten gewinnt.

**Zu den Daten:** Beispieldaten gibt es nicht. Das Turniersystem gibt es nicht mehr; wir bauen Spielfeld und Ball selbst nach und programmieren einige Gegner.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation in Schritten:** Wir bauen das Spiel nach: Ball bewegen, Abprallen prüfen, Schläger bewegen, Punkte zählen.
- **Gerade fortsetzen:** Zwischen zwei Abprallern fliegt der Ball geradlinig. Mit zwei beobachteten Positionen kennt man die Richtung und kann berechnen, wo der Ball die Schlägerspalte erreicht.
- **Spiegeln statt Abprallen simulieren:** Man verlängert die Gerade einfach über den Spielfeldrand hinaus und faltet die Höhe danach zurück („ungefaltet“ rechnen). Das spart die Simulation jedes Abprallers.
- **Rest bei Division (Modulo):** Das Zurückfalten ist ein Rest mit der Periode 2 · 59 = 118.
- **Messfehler:** Die sichtbare Zelle ist auf ganze Zahlen gerundet. Je mehr Beobachtungen man für die Richtung nimmt, desto genauer wird sie.
- **Bewertung von Zügen (Spielbaum, Minimax):** Hier nur in der einfachsten Form: Wir bewerten die drei Schlägerdrittel nach dem Abstand der Landestelle zum Gegner und nehmen das beste.

**Nützliche Funktionen und Bibliotheken**
- `import math`: `math.atan2`, `math.tan`, `math.cos`, `math.sin`, `math.hypot`, `math.radians` für Winkel.
- `random.uniform(a, b)`: Zufallszahl zwischen `a` und `b`; `random.seed(n)`: jedes Mal dieselben Zufallszahlen.
- `round(zahl)`: auf die nächste ganze Zahl runden (so sieht ein Spieler den Ball).
- `zahl % 118`: Rest bei der Division.

## Die Idee in Worten

Die Aufgabe ist offen: Es gibt keine „richtige“ Ausgabe, sondern eine KI, die im Turnier gut abschneidet. Das Turniersystem existiert nicht mehr, also bauen wir uns einen kleinen Spielrahmen und lassen unsere Spieler gegeneinander antreten.

Die Lösungshinweise geben drei Bausteine vor:

1. **Treffen und zurückspielen (defensiv):** Aus den sichtbaren Ballzellen die Flugrichtung schätzen und berechnen, in welcher Höhe der Ball den eigenen Schläger erreicht. Der Schläger bewegt sich dorthin.
2. **Schläger positionieren:** Nach dem Zurückspielen den Schläger dorthin bewegen, wo der Ball auf der Gegenseite auftrifft. Der Ball kommt mit gleichem Winkel zurück, wenn der Gegner ihn mit der Mitte annimmt.
3. **Gewinnen (offensiv):** Das Drittel des Schlägers wählen, mit dem der Ball so weit wie möglich vom Gegner weg landet.

Der Plan:

1. **Spielfeld und Ball** nachbauen (Schritt 1), dabei die Regeln für Wände und Schlägerdrittel.
2. **Spielablauf** mit Halbzeiten, Einwurf und Punkten (Schritt 2) und einfache Gegner (Schritt 3).
3. **Vorhersage** des Auftreffpunkts (Schritt 4), **Schlägerdrittel wählen** (Schritt 5).
4. **Turnier** jeder gegen jeden (Schritt 6).

## Schritt 1: Spielfeld und Ball

Die Hinweise beschreiben das Spielfeld genauer: Die Zellenmitten liegen auf den Punkten `(0 | 0)` bis `(64 | 59)`, `y` wächst nach oben. Die Schläger stehen in den Spalten `x = 0` und `x = 64`. Der Ball hat eine **echte Position** (Kommazahlen) und eine **sichtbare Position**: die gerundete Zelle. Die Geschwindigkeit ist am Anfang 1, bei jedem Abprallen (Bande oder Schläger) wird sie mit 1,05 multipliziert.

Wir legen folgende **Annahmen** fest (das Turniersystem lässt sich nicht mehr nachlesen):

- Der Ball prallt an `y = 0` und `y = 59` ab. Erreicht er `x = 0` oder `x = 64`, prallt er ab, wenn der Schläger dort steht (Schlägerzellen `sy` bis `sy + 5`, also die Höhen von `sy − 0,5` bis `sy + 5,5`); sonst bekommt der Gegner einen Punkt.
- Die Drittel des Schlägers sind je zwei Zellen hoch: unten, Mitte, oben.
- Nach einem Punkt beginnt der Einwurf wieder mit Tempo 1 (sonst wäre der Ball irgendwann nicht mehr zu erreichen). Er startet neben dem Schläger des Einwerfenden auf Höhe 29,5 mit einem Winkel zwischen −70° und +70° zur Waagerechten.
- Eine Halbzeit hat 500 Schritte, die Schläger starten bei `sy = 27` (Zellen 27 bis 32).

Weil der Ball in einem Schritt auch mehrfach abprallen kann, bewegt `bewege_ball` ihn in Teilstücken bis zum nächsten Ereignis (Bande oder Schlägerspalte).

In [ ]:
import math
import random

BREITE, HOEHE, SCHLAEGER = 65, 60, 6
MAX_WINKEL = math.radians(70)
SCHRITTE = 500

In [ ]:
def einwurf(von_links):
    winkel = random.uniform(-MAX_WINKEL, MAX_WINKEL)
    ball = {'x': 1.0, 'y': 29.5, 'vx': math.cos(winkel), 'vy': math.sin(winkel)}
    if not von_links:
        ball['x'] = BREITE - 2.0
        ball['vx'] = -ball['vx']
    return ball

**Abprallen am Schläger.** Zuerst wird wie bei einer Bande gespiegelt (`vx` dreht das Vorzeichen, beide Teile werden mit 1,05 multipliziert). Trifft der Ball das obere oder untere Drittel, wählen wir für den neuen Winkel eine Zufallszahl zwischen dem regulären Winkel und ±70°; das Tempo bleibt dabei gleich. Dazu rechnen wir `vx, vy` mit `atan2`, `cos` und `sin` in einen Winkel und zurück. `rel` ist die Höhe des Balls über der Unterkante des Schlägers (0 bis 6): unter 2 ist das untere Drittel, ab 4 das obere.

In [ ]:
def prall_schlaeger(ball, sy):
    ball['vx'] = -ball['vx'] * 1.05
    ball['vy'] = ball['vy'] * 1.05
    rel = ball['y'] - (sy - 0.5)
    if rel >= 4 or rel < 2:
        tempo = math.hypot(ball['vx'], ball['vy'])
        regulaer = math.atan2(ball['vy'], abs(ball['vx']))
        if rel >= 4:
            neu = random.uniform(regulaer, max(regulaer, MAX_WINKEL))
        else:
            neu = random.uniform(min(regulaer, -MAX_WINKEL), regulaer)
        vorzeichen = 1 if ball['vx'] > 0 else -1
        ball['vx'] = vorzeichen * tempo * math.cos(neu)
        ball['vy'] = tempo * math.sin(neu)

In [ ]:
def bewege_ball(ball, sl, sr):
    """Bewegt den Ball um einen Schritt. Rückgabe: None, 'links' oder 'rechts' (Seite, die den Punkt verliert)."""
    rest = 1.0
    while True:
        if ball['vy'] > 0:
            t_wand = (HOEHE - 1 - ball['y']) / ball['vy']
        elif ball['vy'] < 0:
            t_wand = ball['y'] / -ball['vy']
        else:
            t_wand = 1e18
        if ball['vx'] < 0:
            t_seite = ball['x'] / -ball['vx']
        else:
            t_seite = (BREITE - 1 - ball['x']) / ball['vx']
        if min(t_wand, t_seite) >= rest:
            ball['x'] += ball['vx'] * rest
            ball['y'] += ball['vy'] * rest
            return None
        t = min(t_wand, t_seite)
        ball['x'] += ball['vx'] * t
        ball['y'] += ball['vy'] * t
        rest -= t
        if t_wand <= t_seite:
            ball['y'] = 0.0 if ball['vy'] < 0 else HOEHE - 1.0
            ball['vx'] *= 1.05
            ball['vy'] = -ball['vy'] * 1.05
        else:
            nach_links = ball['vx'] < 0
            sy = sl if nach_links else sr
            ball['x'] = 0.0 if nach_links else BREITE - 1.0
            if sy - 0.5 <= ball['y'] <= sy + SCHLAEGER - 0.5:
                prall_schlaeger(ball, sy)
            else:
                return 'links' if nach_links else 'rechts'

**Test:** Wir treffen einen Ball mit `vx = −1`, `vy = 0,5` (regulärer Ausfallswinkel ca. 26,6°) im Mitteldrittel, im oberen und im unteren Drittel und schauen, welche Winkel herauskommen.

In [ ]:
def teste_drittel(hoehe, anzahl=1000):
    """Winkel (in Grad) nach dem Abprallen am Schläger bei sy = 27 für einen Treffer in der Höhe `hoehe`."""
    winkel = []
    for i in range(anzahl):
        ball = {'x': 0.0, 'y': hoehe, 'vx': -1.0, 'vy': 0.5}
        prall_schlaeger(ball, 27)
        winkel.append(math.degrees(math.atan2(ball['vy'], ball['vx'])))
    return round(min(winkel), 1), round(max(winkel), 1)


random.seed(1)
print('Mitte  (Höhe 29,5):', teste_drittel(29.5))
print('oben   (Höhe 32,0):', teste_drittel(32.0))
print('unten  (Höhe 27,0):', teste_drittel(27.0))

Mitte  (Höhe 29,5): (26.6, 26.6)
oben   (Höhe 32,0): (26.6, 69.9)
unten  (Höhe 27,0): (-69.9, 26.4)


In der Mitte kommt immer derselbe Winkel (26,6°) heraus, oben liegen die Winkel zwischen 26,6° und 70°, unten zwischen −70° und 26,6°. Das passt zur Aufgabe.

**Sichtbar und echt:** Ein Spieler sieht nur die gerundeten Zellen. Hier ein Einwurf von links, die ersten Schritte:

In [ ]:
random.seed(3)
ball = einwurf(True)
print('Schritt  echte Position    sichtbare Zelle')
for schritt in range(8):
    print('%7d  (%5.2f | %5.2f)   (%d | %d)' % (schritt, ball['x'], ball['y'], round(ball['x']), round(ball['y'])))
    bewege_ball(ball, 27, 27)

Schritt  echte Position    sichtbare Zelle
      0  ( 1.00 | 29.50)   (1 | 30)
      1  ( 1.80 | 28.90)   (2 | 29)
      2  ( 2.60 | 28.31)   (3 | 28)
      3  ( 3.41 | 27.71)   (3 | 28)
      4  ( 4.21 | 27.11)   (4 | 27)
      5  ( 5.01 | 26.51)   (5 | 27)
      6  ( 5.81 | 25.92)   (6 | 26)
      7  ( 6.61 | 25.32)   (7 | 25)


## Schritt 2: Der Spielablauf

Eine **Halbzeit** läuft so: In jedem Schritt wird zuerst gefragt, wie sich die Spieler bewegen, dann bewegt sich der Ball. Fällt ein Punkt, wird neu eingeworfen (abwechselnd links und rechts), und beide Schläger gehen in die Mitte zurück.

Jeder Spieler wird durch den **Namen** seiner Strategie (Text) beschrieben; die Funktion `zug(name, sicht, mem)` liefert −1 (Schläger nach unten), 0 oder +1 (nach oben). `sicht` enthält, was der Spieler sieht: die Ballzelle sowie die Position des eigenen und des gegnerischen Schlägers. `mem` ist ein Wörterbuch, in dem sich der Spieler Dinge über die Schritte hinweg merken kann.

Damit alle Strategien nur den linken Schläger steuern müssen (so verlangt es die Aufgabe), bekommt der rechte Spieler das **gespiegelte** Feld zu sehen: `x` wird durch `64 − x` ersetzt.

In [ ]:
def halbzeit(links, rechts, links_wirft):
    sl = sr = 27
    ball = einwurf(links_wirft)
    pl = pr = 0
    mem_l = {'verlauf': []}
    mem_r = {'verlauf': []}
    wirft_links = links_wirft
    for schritt in range(SCHRITTE):
        bx, by = round(ball['x']), round(ball['y'])
        zl = zug(links, {'ball': (bx, by), 'eigen': sl, 'gegner': sr}, mem_l)
        zr = zug(rechts, {'ball': (BREITE - 1 - bx, by), 'eigen': sr, 'gegner': sl}, mem_r)
        sl = min(max(sl + zl, 0), HOEHE - SCHLAEGER)
        sr = min(max(sr + zr, 0), HOEHE - SCHLAEGER)
        verlierer = bewege_ball(ball, sl, sr)
        if verlierer is not None:
            if verlierer == 'links':
                pr += 1
            else:
                pl += 1
            wirft_links = not wirft_links
            ball = einwurf(wirft_links)
            sl = sr = 27
    return pl, pr

## Schritt 3: Einfache Gegner

Drei Gegner, die wir später schlagen wollen:

- `steht`: bewegt den Schläger nie.
- `zufall`: bewegt den Schläger in jedem Schritt zufällig nach oben, unten oder gar nicht.
- `folgt`: bewegt den Schläger immer zur Höhe der sichtbaren Ballzelle. Das ist die Strategie, die die Hinweise ausdrücklich als zu einfach bezeichnen („mehr tun als einfach der Höhe des Balles zu folgen“).

Weil wir mehrere Versionen von `zug` nacheinander bauen, steht hier zunächst `zug_einfach`; `zug` ruft sie nur auf (später wird `zug` ersetzt).

In [ ]:
def zug_einfach(name, sicht, mem):
    sy = sicht['eigen']
    if name == 'zufall':
        return random.choice([-1, 0, 1])
    if name == 'folgt':
        ziel = sicht['ball'][1] - 2.5          # Schlägermitte auf Ballhöhe
        if ziel > sy:
            return 1
        if ziel < sy:
            return -1
    return 0                                    # 'steht' (und alles andere)


def zug(name, sicht, mem):
    return zug_einfach(name, sicht, mem)


random.seed(2)
for paar in [('steht', 'steht'), ('folgt', 'steht'), ('zufall', 'steht'), ('folgt', 'zufall')]:
    print('%-6s gegen %-6s ->' % paar, halbzeit(paar[0], paar[1], True))

steht  gegen steht  -> (1, 1)
folgt  gegen steht  -> (3, 0)
zufall gegen steht  -> (3, 2)
folgt  gegen zufall -> (3, 0)


Die Zahlen sind klein, weil ein Punkt erst fällt, wenn ein Spieler den Ball verfehlt. `folgt` gewinnt gegen `steht` und `zufall` (in diesem Lauf 3 : 0), während `steht` und `zufall` gegeneinander nur der Zufall der Einwürfe und Abpraller entscheidet. Das genaue Ergebnis hängt vom Zufall ab; im Turnier (Schritt 6) mitteln wir über viele Spiele.

## Schritt 4: Wo trifft der Ball auf? (Vorhersage)

Die Hinweise: Die Flugbahn ist eine Gerade; die Höhe, in der der Ball die Spalte `x = 0` erreicht, ergibt sich aus der Geraden durch zwei Positionen. Wände muss man dabei nicht simulieren. Man rechnet so, als gäbe es sie nicht („ungefaltet“) und faltet das Ergebnis zurück:

- Die Wände liegen bei `y = 0` und `y = 59`, nach 118 Zellen wiederholt sich das Muster.
- Von `0` bis `59` ist die Höhe unverändert, von `59` bis `118` wird sie gespiegelt: `118 − y`.

Genau das macht `falten`. Beim Ungefalteten bleibt auch die Beschleunigung ohne Einfluss: Bei jedem Abprall werden `vx` und `vy` mit demselben Faktor multipliziert, die Richtung der Geraden ändert sich also nicht.

**Richtung schätzen:** Eine sichtbare Zelle ist um bis zu ±0,5 falsch. Nimmt man nur zwei aufeinanderfolgende Zellen, ist `vy` sehr ungenau. Wir merken uns deshalb die letzten (höchstens 30) Zellen seit dem letzten Abprallen und teilen den Unterschied zwischen der ersten und der letzten durch die Zahl der Schritte. Ein Abprallen an Bande, Schläger oder ein neuer Einwurf beendet den Verlauf (`merke` leert ihn dann).

In [ ]:
def vorzeichen(zahl):
    if zahl > 0:
        return 1
    if zahl < 0:
        return -1
    return 0


def falten(y):
    """Spiegelt eine ungefaltete y-Koordinate zurück ins Spielfeld (Wände bei 0 und 59)."""
    y = y % (2 * (HOEHE - 1))
    if y > HOEHE - 1:
        y = 2 * (HOEHE - 1) - y
    return y

In [ ]:
def schaetze(verlauf):
    """Geschwindigkeit aus den sichtbaren Zellen (letzte höchstens 30)."""
    n = len(verlauf)
    return (verlauf[-1][0] - verlauf[0][0]) / (n - 1), (verlauf[-1][1] - verlauf[0][1]) / (n - 1)


def treffpunkt(verlauf, ziel_x):
    """Höhe, in der der Ball die Spalte ziel_x erreicht (None, wenn er sich nicht darauf zu bewegt)."""
    if len(verlauf) < 2:
        return None
    vx, vy = schaetze(verlauf)
    x, y = verlauf[-1]
    if (ziel_x - x) * vx <= 0.05:
        return None
    return falten(y + vy * (ziel_x - x) / vx)

In [ ]:
def merke(mem, pos):
    """Pflegt den Verlauf der sichtbaren Ballzellen; bei Wand, Schläger oder Einwurf beginnt er neu."""
    verlauf = mem['verlauf']
    if verlauf:
        dx = pos[0] - verlauf[-1][0]
        dy = pos[1] - verlauf[-1][1]
        if abs(dx) > 20 or pos[1] <= 1 or pos[1] >= HOEHE - 2:
            verlauf.clear()
        elif len(verlauf) >= 2:
            ax = verlauf[-1][0] - verlauf[-2][0]
            ay = verlauf[-1][1] - verlauf[-2][1]
            if vorzeichen(dx) * vorzeichen(ax) < 0 or (abs(dy) >= 1 and abs(ay) >= 1 and vorzeichen(dy) * vorzeichen(ay) < 0):
                verlauf.clear()
    verlauf.append(pos)
    del verlauf[:-30]

**Test der Vorhersage.** Wir werfen den Ball 200-mal von rechts ein (ohne Schläger), geben die sichtbaren Zellen an `merke` weiter und halten fest, was `treffpunkt` vorhersagt, wenn der Ball in Spaltenmitte (`x ≤ 32`) ist. Zum Vergleich steht die tatsächliche Höhe, in der der Ball die Spalte 0 erreicht.

In [ ]:
def pruefe_vorhersage(anzahl):
    fehler = []
    for i in range(anzahl):
        ball = einwurf(False)
        mem = {'verlauf': []}
        vorhergesagt = None
        while True:
            merke(mem, (round(ball['x']), round(ball['y'])))
            if vorhergesagt is None and ball['x'] <= 32:
                vorhergesagt = treffpunkt(mem['verlauf'], 0)
            if bewege_ball(ball, -100, -100) is not None:      # Schläger weit weg: der Ball geht durch
                break
        if vorhergesagt is not None:
            fehler.append(abs(vorhergesagt - ball['y']))
    return fehler


random.seed(4)
fehler = pruefe_vorhersage(200)
print('Vorhersagen:', len(fehler))
print('mittlerer Fehler: %.2f Zellen' % (sum(fehler) / len(fehler)))
print('Fehler höchstens 2 Zellen (Schläger fängt den Ball noch mit der Mitte):', sum(1 for f in fehler if f <= 2))
print('größter Fehler: %.1f Zellen' % max(fehler))

Vorhersagen: 200
mittlerer Fehler: 1.58 Zellen
Fehler höchstens 2 Zellen (Schläger fängt den Ball noch mit der Mitte): 153
größter Fehler: 23.5 Zellen


Im Mittel liegt die Vorhersage etwa 1,6 Zellen daneben, in rund drei Viertel der Fälle höchstens 2 Zellen (dann trifft der Schläger den Ball noch), und das schon in der Mitte des Feldes. Einzelne Ausreißer gibt es trotzdem: Wenn der Ball kurz vorher abgeprallt ist, ist der Verlauf kurz und die Richtung ungenau. Das ist nicht so schlimm, denn wir rechnen in jedem Schritt neu: Je näher der Ball kommt, desto länger wird der Verlauf und desto besser wird die Schätzung.

## Schritt 5: Wo steht der Schläger? Und welches Drittel nehmen wir?

Mit der Vorhersage ist die **Verteidigung** einfach: Der Schläger bewegt sich so, dass seine Mitte (`sy + 2,5`) auf der vorhergesagten Höhe liegt. Kommt der Ball nicht auf uns zu, sondern fliegt er zum Gegner, sagen wir seinen Auftreffpunkt in Spalte 64 voraus und stellen uns dorthin (siehe Hinweise: mit Winkel 0° kommt der Ball genau von dort zurück). Ist gar nichts bekannt (kurz nach dem Einwurf), bleiben wir bei der letzten Zielhöhe.

Für die **Offensive** zählt der Zeitpunkt, an dem wir den Ball annehmen: Wir können entscheiden, welches Drittel des Schlägers den Ball trifft. Für jedes Drittel schätzen wir den Ausfallswinkel (mittleres Drittel: der reguläre Winkel `w0`; oberes: Mitte zwischen `w0` und 70°; unteres: Mitte zwischen −70° und `w0`) und rechnen aus, in welcher Höhe der Ball bei `x = 64` ankäme. Wir wählen das Drittel mit der größten Entfernung zur Mitte des gegnerischen Schlägers. Ein Drittel, das am Spielfeldrand nicht mehr erreichbar ist (der Schläger kann nicht weiter nach oben oder unten), scheidet aus.

Wir zielen nicht auf die äußerste Zeile des Drittels, sondern etwas weiter innen (relative Höhe 4,5 bzw. 1,5), damit kleine Fehler der Vorhersage das Drittel nicht sofort verfehlen lassen.

In [ ]:
def angriff_ziel(verlauf, ty, gegner_sy):
    """Wählt das Drittel, mit dem man den Ball bei Höhe ty annimmt, und gibt den passenden Schlägerplatz zurück."""
    vx, vy = schaetze(verlauf)
    w0 = math.atan2(vy, abs(vx))
    grad70 = MAX_WINKEL
    optionen = [(4.0, 4, 6, (w0 + grad70) / 2),      # oberes Drittel
                (2.5, 2, 4, w0),                      # mittleres Drittel
                (1.0, 0, 2, (w0 - grad70) / 2)]       # unteres Drittel
    bestes = None
    for versatz, von, bis, winkel in optionen:
        sy = min(max(ty - versatz, 0), HOEHE - SCHLAEGER)
        rel = ty - (sy - 0.5)
        if not (von <= rel < bis):
            continue                                   # dieses Drittel ist am Rand nicht erreichbar
        landung = falten(ty + math.tan(winkel) * (BREITE - 1))
        abstand = abs(landung - (gegner_sy + 2.5))
        if bestes is None or abstand > bestes[0]:
            bestes = (abstand, sy)
    return bestes[1]

Die Wirkung an zwei Beispielen: Der Ball kommt waagerecht auf Höhe 30 (`verlauf` mit gleichbleibender Höhe). Steht der Gegner oben (`gegner_sy = 40`), nehmen wir den Ball mit dem unteren Drittel; steht er unten (`gegner_sy = 5`), mit dem oberen.

In [ ]:
def drittel_von(ty, sy):
    rel = ty - (sy - 0.5)
    if rel < 2:
        return 'unteres Drittel'
    if rel < 4:
        return 'mittleres Drittel'
    return 'oberes Drittel'


verlauf = [(20, 30), (19, 30), (18, 30), (17, 30)]
for gegner in [40, 5]:
    sy = angriff_ziel(verlauf, 30, gegner)
    print('Gegner bei sy =', gegner, '-> unser sy =', sy, '->', drittel_von(30, sy))

Gegner bei sy = 40 -> unser sy = 29.0 -> unteres Drittel
Gegner bei sy = 5 -> unser sy = 26.0 -> oberes Drittel


Jetzt kommen alle Strategien in `zug`: `vorhersage` nimmt immer die Mitte, `angriff` wählt das Drittel mit `angriff_ziel`. Das Ziel wird mit `round` auf eine ganze Zelle gerundet und der Schläger einen Schritt dorthin bewegt.

In [ ]:
def zug(name, sicht, mem):
    """Gibt -1 (nach unten), 0 oder +1 (nach oben) zurück."""
    sy = sicht['eigen']
    if name == 'steht':
        return 0
    if name == 'zufall':
        return random.choice([-1, 0, 1])
    if name == 'folgt':
        return vorzeichen(sicht['ball'][1] - (sy + 2.5))
    merke(mem, sicht['ball'])
    verlauf = mem['verlauf']
    ty = treffpunkt(verlauf, 0)
    if ty is not None:
        if name == 'angriff':
            ziel = angriff_ziel(verlauf, ty, sicht['gegner'])
        else:
            ziel = ty - 2.5
    else:
        weg = treffpunkt(verlauf, BREITE - 1)
        if weg is not None:
            ziel = weg - 2.5
        else:
            ziel = mem.get('ziel', 27)
    ziel = min(max(ziel, 0), HOEHE - SCHLAEGER)
    mem['ziel'] = ziel
    return vorzeichen(round(ziel) - sy)

## Schritt 6: Turnier

Jeder spielt gegen jeden. Ein **Spiel** besteht aus zwei Halbzeiten; in der ersten wirft der erste Spieler zuerst ein, in der zweiten der zweite (und er steht dann links). Wir spielen jede Paarung `ANZAHL`-mal, summieren die Punkte und zählen, wer die meisten Spiele gewonnen hat.

In [ ]:
def spiel(a, b):
    """Zwei Halbzeiten: in der ersten wirft a zuerst ein, in der zweiten b. Rückgabe: Punkte von a und von b."""
    pa1, pb1 = halbzeit(a, b, True)
    pb2, pa2 = halbzeit(b, a, True)
    return pa1 + pa2, pb1 + pb2


namen = ['steht', 'zufall', 'folgt', 'vorhersage', 'angriff']
ANZAHL = 40
random.seed(2014)
punkte = {}
sieg = {}
for i in range(len(namen)):
    for j in range(i + 1, len(namen)):
        a = namen[i]
        b = namen[j]
        punkte[(a, b)] = [0, 0]
        for k in range(ANZAHL):
            pa, pb = spiel(a, b)
            punkte[(a, b)][0] += pa
            punkte[(a, b)][1] += pb
            if pa > pb:
                sieg[a] = sieg.get(a, 0) + 1
            if pb > pa:
                sieg[b] = sieg.get(b, 0) + 1

print('Punkte (Zeile : Spalte) über %d Spiele' % ANZAHL)
print('%-11s' % '', *['%11s' % n for n in namen])
for a in namen:
    zeile = []
    for b in namen:
        if a == b:
            zeile.append('%11s' % '-')
        elif (a, b) in punkte:
            zeile.append('%11s' % ('%d : %d' % (punkte[(a, b)][0], punkte[(a, b)][1])))
        else:
            zeile.append('%11s' % ('%d : %d' % (punkte[(b, a)][1], punkte[(b, a)][0])))
    print('%-11s' % a, *zeile)
print()
print('Gewonnene Spiele (von %d):' % (ANZAHL * (len(namen) - 1)))
for n in namen:
    print('  %-11s %3d' % (n, sieg.get(n, 0)))

Punkte (Zeile : Spalte) über 40 Spiele
                  steht      zufall       folgt  vorhersage     angriff
steht                 -   177 : 200     4 : 237     0 : 241     0 : 235
zufall        200 : 177           -     3 : 235     0 : 250     1 : 257
folgt           237 : 4     235 : 3           -      3 : 38      4 : 53
vorhersage      241 : 0     250 : 0      38 : 3           -       3 : 7
angriff         235 : 0     257 : 1      53 : 4       7 : 3           -

Gewonnene Spiele (von 160):
  steht        11
  zufall       18
  folgt        81
  vorhersage  110
  angriff     118


Was man sieht (die genauen Zahlen hängen vom Zufall ab, die Reihenfolge ändert sich kaum):

- `vorhersage` und `angriff` schlagen `steht` und `zufall` mit rund 240 zu 0 Punkten in 40 Spielen: Sie verfehlen praktisch nie einen Ball, die Gegner fast immer.
- `folgt` verliert gegen beide klar (38 : 3 und 53 : 4), obwohl er `steht` und `zufall` schlägt. Der Ball ändert nach dem Abprallen am Schläger seinen Winkel und beschleunigt; wer nur der sichtbaren Ballhöhe folgt, kommt zu spät.
- Zwischen `vorhersage` und `angriff` ist der Unterschied klein (7 : 3 Punkte, 118 zu 110 gewonnene Spiele). Die Offensive hilft also etwas, gegen die Gegner mit einer guten Verteidigung fallen aber kaum Punkte. Ob das Drittelwählen wirklich besser ist, zeigt erst ein größeres Turnier (mehr Spiele pro Paar, mehr verschiedene Gegner).
- Die Zufallsgegner `steht` und `zufall` sind praktisch gleich stark; den Unterschied in ihren Siegen (11 gegen 18) sollte man nicht überbewerten.

## Zum Schluss: MiniZinc?

Wir haben MiniZinc hier nicht eingesetzt. Die Aufgabe hat keine Eingabedaten: Man schreibt eine Strategie, die in Echtzeit gegen unbekannte Gegner spielt, mit zufälligen Abprallwinkeln. Das ist eine Simulation mit Zufall und kein Rechenproblem mit festen Eingaben; ein Constraint-Modell hätte nichts zu lösen, und wir haben deshalb auch nichts zu messen.

## Für Neugierige

- **Ausgleichsgerade:** Statt der ersten und letzten Zelle des Verlaufs kann man durch alle Zellen eine Ausgleichsgerade legen (Methode der kleinsten Quadrate). Das senkt den Vorhersagefehler.
- **Minimax:** Die Hinweise beschreiben, wie man die Wahl der Drittel als Spielbaum betrachtet: Ein Zug ist gewonnen, wenn der Gegner den Ball danach nicht mehr erreichen kann (die Flugzeit des Balls ist kleiner als die Zeit, die der Gegner-Schläger braucht). Wir haben nur einen Zug vorausgeschaut und die Entfernung zum Gegner als Bewertung genommen.
- **Gegner beobachten:** Wenn der Gegner immer die Mitte nimmt oder immer zur Mitte zurückläuft, lohnt es sich, das zu merken und den Ball dorthin zu spielen, wo er am längsten braucht.
- **Echte Zufallsrichtung:** Die Hinweise erwähnen, dass das Tempo vom Betrag her wächst. Wer das Turniersystem noch hätte, würde die Wachstumsrate (1,05 pro Abprall) aus den Daten schätzen.